# DSCMFNet — Google Colab Training + Testing

This notebook runs end-to-end training and evaluation of **DSCMFNet** (Dual-Stream
Cross-Modal Fusion Network) for Early Gastric Cancer segmentation.

**Assumptions:**
- Repository is available in Google Drive **or** cloned from GitHub (see Cell 2).
- Processed dataset (`processed_data/`) is stored in Google Drive.
- Checkpoints, TensorBoard logs, and prediction previews are written back to Drive.

**Cells overview:**
1. Check GPU
2. Mount Google Drive and enter the repo (or clone from GitHub)
3. Install dependencies
4. Configure paths and training arguments
5. Smoke-test override (optional)
6. Optional preprocessing
7. Launch training
8. Inspect saved outputs
9. Load checkpoint and evaluate
10. Save prediction previews to Drive
11. TensorBoard

## 1. Check GPU availability

Training on CPU is extremely slow. In Colab go to
**Runtime → Change runtime type** and select a **T4 GPU** (or higher) before
running this notebook.

In [ ]:
import torch

if torch.cuda.is_available():
    _props = torch.cuda.get_device_properties(0)
    print(f'GPU : {_props.name}')
    print(f'VRAM: {_props.total_memory / 1e9:.1f} GB')
    DEVICE = 'cuda'
else:
    print('WARNING: No GPU detected. Training will be very slow on CPU.')
    DEVICE = 'cpu'

print(f'PyTorch : {torch.__version__}')
print(f'Device  : {DEVICE}')

## 2. Mount Google Drive and enter the repo

**Option A (default):** Repository is already synced to Google Drive — update
`DRIVE_REPO_PATH` if needed.

**Option B:** Clone directly from GitHub — set `USE_DRIVE = False` and fill in
`GITHUB_REPO`.

In [ ]:
from google.colab import drive
from pathlib import Path
import os

# ── Option A: load from Google Drive ─────────────────────────────────────────
USE_DRIVE = True   # set False to use Option B (GitHub clone) instead

if USE_DRIVE:
    drive.mount('/content/drive')
    DRIVE_REPO_PATH = Path('/content/drive/MyDrive/early_gastric_cancer_detection')
    assert DRIVE_REPO_PATH.exists(), f'Repo path not found: {DRIVE_REPO_PATH}'
    os.chdir(DRIVE_REPO_PATH)
else:
    # ── Option B: clone from GitHub ──────────────────────────────────────────
    GITHUB_REPO = 'https://github.com/<YOUR_ORG>/early_gastric_cancer_detection.git'
    CLONE_DIR   = Path('/content/egc_repo')
    if not CLONE_DIR.exists():
        os.system(f'git clone {GITHUB_REPO} {CLONE_DIR}')
    os.chdir(CLONE_DIR)

print('Working directory:', Path.cwd())

## 3. Install dependencies

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)
# Extras that may not be pre-installed in all Colab runtime versions
subprocess.run([
    sys.executable, '-m', 'pip', 'install',
    'tensorboard', 'matplotlib', 'pandas',
    'timm',            # PVTv2-B2 backbone
    'albumentations',  # augmentation pipeline
], check=True)

## 4. Configure paths and experiment settings

All important inputs/outputs point into **Google Drive** so they survive runtime
restarts.

In [ ]:
from pathlib import Path
import json

# ── Google Drive locations ────────────────────────────────────────────────────
DATA_ROOT   = Path('/content/drive/MyDrive/egc_data/processed_data')
OUTPUT_ROOT = Path('/content/drive/MyDrive/egc_runs/dscmfnet_colab_run')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Optional: raw data location for preprocessing
RAW_ROOT = Path('/content/drive/MyDrive/egc_data/raw_data')

# ── Training configuration ────────────────────────────────────────────────────
TRAIN_CONFIG = {
    'device':        DEVICE,      # set automatically by the GPU check cell above
    'phase':         1,           # 1 = same-scale, 2 = cross-scale
    'model':         'dscmfnet',  # nbi_only | wli_only | early_fusion | dscmfnet
    'fusion_mode':   'concat',    # concat | cmfim | cmfim_no_sam | cmfim_no_boundary
    'epochs':        50,
    'batch_size':    4,
    'img_size':      352,
    'lr':            1e-4,
    'weight_decay':  1e-4,
    'freeze_stages': 2,
    'lambda_bdy':    0.5,
    'alpha_ds':      0.3,
    'lambda_wli':    0.0,
    'kfold':         5,
    'seed':          42,
    'num_workers':   2,
    'amp':           True,
    'pretrained':    True,
    'print_freq':    5,
}

assert DATA_ROOT.exists(), f'Data root not found: {DATA_ROOT}'
print('DATA_ROOT  =', DATA_ROOT)
print('OUTPUT_ROOT=', OUTPUT_ROOT)
print(json.dumps(TRAIN_CONFIG, indent=2))

## 5. Smoke-test mode (optional)

Set `SMOKE_TEST = True` to do a quick 2-epoch, 2-fold dry run before committing
to a full training session. Useful for verifying that paths, data, and the
model all work correctly.

In [ ]:
SMOKE_TEST = False  # ← set True for a fast sanity check

if SMOKE_TEST:
    TRAIN_CONFIG.update({
        'epochs':     2,
        'kfold':      2,
        'batch_size': 2,
        'amp':        False,  # off for easier debugging
        'print_freq': 1,
    })
    print('Smoke-test mode — config:')
    print(json.dumps(TRAIN_CONFIG, indent=2))
else:
    print('Full training mode.')

## 6. Optional preprocessing

Run only if `RAW_ROOT` contains raw Olympus/LabelMe data and `processed_data/`
has not been built yet.

In [ ]:
import subprocess

RUN_PREPROCESS = False

if RUN_PREPROCESS:
    preprocess_cmd = [
        'python', 'preprocess.py',
        '--raw_root', str(RAW_ROOT),
        '--out_root', str(DATA_ROOT),
    ]
    print('Running:', ' '.join(preprocess_cmd))
    subprocess.run(preprocess_cmd, check=True)
else:
    print('Skipping preprocessing (RUN_PREPROCESS=False).')

## 7. Launch training

Writes **checkpoints**, **TensorBoard logs**, and **results.json** to
`OUTPUT_ROOT` on Google Drive.

In [ ]:
import shlex
import subprocess

cmd = [
    'python', 'train.py',
    '--data_root',     str(DATA_ROOT),
    '--output',        str(OUTPUT_ROOT),
    '--device',        TRAIN_CONFIG['device'],
    '--phase',         str(TRAIN_CONFIG['phase']),
    '--model',         TRAIN_CONFIG['model'],
    '--fusion_mode',   TRAIN_CONFIG['fusion_mode'],
    '--epochs',        str(TRAIN_CONFIG['epochs']),
    '--batch_size',    str(TRAIN_CONFIG['batch_size']),
    '--img_size',      str(TRAIN_CONFIG['img_size']),
    '--lr',            str(TRAIN_CONFIG['lr']),
    '--weight_decay',  str(TRAIN_CONFIG['weight_decay']),
    '--freeze_stages', str(TRAIN_CONFIG['freeze_stages']),
    '--lambda_bdy',    str(TRAIN_CONFIG['lambda_bdy']),
    '--alpha_ds',      str(TRAIN_CONFIG['alpha_ds']),
    '--lambda_wli',    str(TRAIN_CONFIG['lambda_wli']),
    '--kfold',         str(TRAIN_CONFIG['kfold']),
    '--seed',          str(TRAIN_CONFIG['seed']),
    '--num_workers',   str(TRAIN_CONFIG['num_workers']),
    '--print_freq',    str(TRAIN_CONFIG['print_freq']),
]

if TRAIN_CONFIG['amp']:
    cmd.append('--amp')
cmd.append('--pretrained' if TRAIN_CONFIG['pretrained'] else '--no-pretrained')

print('Command:\n' + ' '.join(shlex.quote(x) for x in cmd))
subprocess.run(cmd, check=True)

## 8. Inspect saved outputs

In [ ]:
from pathlib import Path
import json

print('Top-level output files:')
for p in sorted(OUTPUT_ROOT.glob('*')):
    print('-', p)

results_path = OUTPUT_ROOT / 'results.json'
if results_path.exists():
    with open(results_path, 'r', encoding='utf-8') as fh:
        results = json.load(fh)
    print('\nSummary from results.json:')
    print(json.dumps(results.get('summary', {}), indent=2))
else:
    print('results.json not found yet.')

## 9. Load a checkpoint and run evaluation

Reuses the repository's Python modules directly so you can evaluate a saved
model without starting a new training run.

In [ ]:
import json
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader

from train import build_model, model_forward, validate
from data.dataloader import (
    EGCPhase1Dataset,
    build_kfold_loaders as build_phase1_loaders,
    find_pairs,
)
from data.dataloader_crossscale import (
    EGCPhase2Dataset,
    build_kfold_loaders as build_phase2_loaders,
    find_crossscale_pairs,
)
from models.losses import DSCMFNetLoss

# ── Select which fold to evaluate ────────────────────────────────────────────
EVAL_FOLD = 0
CHECKPOINT_PATH = OUTPUT_ROOT / f'fold_{EVAL_FOLD}' / 'best.pth'
assert CHECKPOINT_PATH.exists(), f'Checkpoint not found: {CHECKPOINT_PATH}'

ckpt = torch.load(CHECKPOINT_PATH, map_location='cpu')
ckpt_args = ckpt.get('args', {})
print('Checkpoint args:')
print(json.dumps(ckpt_args, indent=2))

# ── Rebuild model from checkpoint args ───────────────────────────────────────
device = torch.device(DEVICE)
model = build_model(
    ckpt_args.get('model', 'dscmfnet'),
    pretrained=False,
    fusion_mode=ckpt_args.get('fusion_mode', 'concat'),
).to(device)
model.load_state_dict(ckpt['state_dict'])
model.eval()
print('Model loaded.')

criterion = DSCMFNetLoss(
    lambda_bdy=ckpt_args.get('lambda_bdy', 0.5),
    alpha_ds=ckpt_args.get('alpha_ds', 0.3),
)

# ── Rebuild validation loader ─────────────────────────────────────────────────
phase       = int(ckpt_args.get('phase', 1))
kfold       = int(ckpt_args.get('kfold', 5))
seed        = int(ckpt_args.get('seed', 42))
img_size    = int(ckpt_args.get('img_size', 352))
batch_size  = int(ckpt_args.get('batch_size', 4))
num_workers = int(ckpt_args.get('num_workers', 2))
lambda_wli  = float(ckpt_args.get('lambda_wli', 0.0))
model_type  = ckpt_args.get('model', 'dscmfnet')
amp         = bool(ckpt_args.get('amp', False)) and device.type == 'cuda'


def _holdout_loader(dataset_cls, find_fn, batch_size, img_size, seed, num_workers):
    """Simple 80/20 holdout split used when kfold == 0."""
    pairs = find_fn(DATA_ROOT)
    n_val = max(1, int(len(pairs) * 0.2))
    rng   = np.random.RandomState(seed)
    perm  = rng.permutation(len(pairs)).tolist()
    val_ds = dataset_cls([pairs[i] for i in perm[-n_val:]], img_size=img_size, augment=False)
    return DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                      num_workers=num_workers, pin_memory=True)


loader_kwargs = dict(
    data_root=DATA_ROOT, fold_idx=EVAL_FOLD, n_folds=kfold,
    batch_size=batch_size, img_size=img_size, seed=seed, num_workers=num_workers,
)

if phase == 1 and kfold > 1:
    _, val_loader = build_phase1_loaders(**loader_kwargs)
elif phase == 2 and kfold > 1:
    _, val_loader = build_phase2_loaders(**loader_kwargs)
elif phase == 1:
    val_loader = _holdout_loader(EGCPhase1Dataset, find_pairs,
                                 batch_size, img_size, seed, num_workers)
else:
    val_loader = _holdout_loader(EGCPhase2Dataset, find_crossscale_pairs,
                                 batch_size, img_size, seed, num_workers)

# ── Run evaluation ────────────────────────────────────────────────────────────
val_metrics = validate(
    model=model, loader=val_loader, criterion=criterion,
    device=device, model_type=model_type, amp=amp, lambda_wli=lambda_wli,
)

print(f'\nValidation metrics — fold {EVAL_FOLD}  [{CHECKPOINT_PATH.name}]')
print(json.dumps(val_metrics, indent=2))

## 10. Save qualitative prediction previews to Google Drive

Exports side-by-side WLI | NBI | Ground Truth | Prediction panels for up to
`max_images` validation samples.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch

PRED_DIR = OUTPUT_ROOT / 'prediction_previews'
PRED_DIR.mkdir(parents=True, exist_ok=True)

_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)[:, None, None]
_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)[:, None, None]


def _denorm(x: np.ndarray) -> np.ndarray:
    """Reverse ImageNet normalisation: (3, H, W) float32 → (H, W, 3) in [0, 1]."""
    return np.clip((x * _STD + _MEAN).transpose(1, 2, 0), 0.0, 1.0)


model.eval()
num_saved = 0
max_images = 8

with torch.no_grad():
    for batch in val_loader:
        wli      = batch['wli'].to(device)
        nbi      = batch['nbi'].to(device)
        mask_np  = (batch['nbi_mask'] if 'nbi_mask' in batch else batch['mask']).cpu().numpy()
        case_ids = batch['case_id']

        # model_forward dispatches correctly for all model variants:
        # nbi_only → model(nbi), wli_only → model(wli), else → model(wli, nbi)
        seg, _bdy, _ds = model_forward(model, wli, nbi, model_type)
        pred_np = torch.sigmoid(seg).cpu().numpy()
        wli_np  = wli.cpu().numpy()
        nbi_np  = nbi.cpu().numpy()

        for i, case_id in enumerate(case_ids):
            fig, axes = plt.subplots(1, 4, figsize=(16, 4))
            axes[0].imshow(_denorm(wli_np[i]))
            axes[0].set_title(f'{case_id} | WLI')
            axes[1].imshow(_denorm(nbi_np[i]))
            axes[1].set_title(f'{case_id} | NBI')
            axes[2].imshow(mask_np[i, 0], cmap='gray', vmin=0, vmax=1)
            axes[2].set_title('Ground Truth')
            axes[3].imshow(pred_np[i, 0], cmap='magma', vmin=0, vmax=1)
            axes[3].set_title('Prediction')
            for ax in axes:
                ax.axis('off')
            fig.tight_layout()

            save_path = PRED_DIR / f'{case_id}_preview.png'
            fig.savefig(save_path, dpi=150, bbox_inches='tight')
            plt.close(fig)
            print('Saved', save_path)

            num_saved += 1
            if num_saved >= max_images:
                break
        if num_saved >= max_images:
            break

print(f'\nSaved {num_saved} preview image(s) to {PRED_DIR}')

## 11. TensorBoard

Loads scalar logs from all fold sub-directories under `OUTPUT_ROOT`. Run this
cell after training has started (or finished).

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {str(OUTPUT_ROOT)}